# FraudeCero · Grupo 6 · 04b · Tablero (capa ORO) e integración
**Proyecto Integrador Big Data – Uniciencia 2026-II** · Integrantes: Juan David Montoya · William Conde Villamizar


**Objetivo:**
1. **Tablero** con 8 visualizaciones (mínimo exigido: 6) sobre la capa oro, cada una con título, unidad y 2 renglones de lectura interpretativa (A3.1). Se publica como **`tablero_fraudecero.html`** (Plotly, interactivo, se abre en cualquier navegador sin servidor).
2. **Integración** (A3.3): exportación de la capa oro a **2 destinos** —archivos CSV/Parquet y base de datos SQLite (consumible por Metabase o Power BI vía ODBC)— y **contrato de datos**.

> El protocolo admite el dashboard nativo de Databricks, Power BI Desktop o Metabase. Por no tener cómputo en Databricks, el tablero se construye en Jupyter con Plotly (biblioteca gratuita y de código abierto); la base SQLite permite conectarlo también a Metabase.

## Configuración común
Esta celda es igual en todos los notebooks. Detecta si se ejecuta en **Google Colab** o en **Jupyter / GitHub Actions**, instala PySpark 3.5 + Delta Lake (OSS) y define:
- `LAKE`: carpeta del Lakehouse (capas `bronce`, `plata`, `oro`).
- `EVID`: carpeta `99_evidencias` (tablas de evidencia, log de monitoreo, tableros HTML).
- `guardar()` / `leer()`: escriben y leen tablas **Delta** (si Delta no está disponible usan Parquet y lo avisan).
- `registrar()`: escribe en el log de monitoreo el tiempo, las filas, el % de nulos y la memoria de cada etapa.

> En Colab cada notebook corre en una máquina distinta, por eso el Lakehouse se respalda en Google Drive al final de cada notebook y se restaura al inicio del siguiente.

In [ ]:
# ===================== CONFIGURACIÓN COMÚN FRAUDECERO =====================
NOTEBOOK = "04b_tablero"
import os, sys, time, json, shutil, datetime, subprocess, hashlib
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub", "psutil", "plotly"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DRIVE = "/content/drive/MyDrive/FraudeCero"
    LAKE = "/content/lakehouse"                      # disco local (rápido)
    LAKE_RESPALDO = f"{BASE_DRIVE}/lakehouse"        # respaldo persistente en Drive
    EVID = f"{BASE_DRIVE}/99_evidencias"
else:
    RAIZ = os.environ.get("FRAUDECERO_RAIZ", os.path.abspath(".."))
    LAKE = os.environ.get("FRAUDECERO_LAKE", os.path.join(RAIZ, "lakehouse"))
    LAKE_RESPALDO = None
    EVID = os.environ.get("FRAUDECERO_EVID", os.path.join(RAIZ, "99_evidencias"))

for carpeta in [LAKE, EVID, f"{EVID}/monitoreo", f"{EVID}/tablas"]:
    os.makedirs(carpeta, exist_ok=True)

def restaurar_lake():
    """En Colab: trae el Lakehouse desde Drive si el disco local está vacío."""
    if LAKE_RESPALDO and os.path.exists(LAKE_RESPALDO) and not os.listdir(LAKE):
        shutil.copytree(LAKE_RESPALDO, LAKE, dirs_exist_ok=True)
        print("Lakehouse restaurado desde Drive")

def respaldar_lake():
    """En Colab: copia el Lakehouse a Drive (sin la zona landing de archivos crudos)."""
    if LAKE_RESPALDO:
        shutil.copytree(LAKE, LAKE_RESPALDO, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("landing"))
        print("Lakehouse respaldado en Drive:", LAKE_RESPALDO)

restaurar_lake()

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

FORMATO = "delta"
try:
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
               .config("spark.driver.memory", "6g")
               .config("spark.sql.shuffle.partitions", "8")
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.range(1).write.format("delta").mode("overwrite").save(f"{LAKE}/_prueba_delta")
except Exception as e:
    print("⚠️ Delta Lake no disponible, se usará Parquet. Detalle:", str(e)[:300])
    FORMATO = "parquet"
    spark = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
             .config("spark.driver.memory", "6g")
             .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def ruta(capa, tabla):
    return f"{LAKE}/{capa}/{tabla}"

def leer(capa, tabla):
    df = spark.read.format(FORMATO).load(ruta(capa, tabla))
    df.createOrReplaceTempView(f"{capa}_{tabla}")
    return df

def guardar(df, capa, tabla, particion=None):
    w = df.write.format(FORMATO).mode("overwrite").option("overwriteSchema", "true")
    if particion:
        w = w.partitionBy(particion)
    w.save(ruta(capa, tabla))
    return leer(capa, tabla)

# ---------- Monitoreo: tiempo, filas, % nulos y memoria por etapa ----------
import psutil, pandas as pd
LOG_MON = f"{EVID}/monitoreo/log_ejecuciones.csv"
ARCHIVO_RUN = f"{EVID}/monitoreo/run_actual.txt"
RUN_ID = os.environ.get("FRAUDECERO_RUN_ID")
if not RUN_ID:
    if NOTEBOOK == "00_ingesta" or not os.path.exists(ARCHIVO_RUN):
        RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")   # cada corrida del pipeline empieza en 00
        with open(ARCHIVO_RUN, "w") as f:
            f.write(RUN_ID)
    else:
        RUN_ID = open(ARCHIVO_RUN).read().strip()
ORIGEN = "github_actions" if os.environ.get("GITHUB_ACTIONS") else ("colab" if EN_COLAB else "jupyter")

def memoria_mb():
    py = psutil.Process(os.getpid()).memory_info().rss / 1024**2
    try:
        rt = spark.sparkContext._jvm.java.lang.Runtime.getRuntime()
        jvm = (rt.totalMemory() - rt.freeMemory()) / 1024**2
    except Exception:
        jvm = 0.0
    return round(py + jvm, 1)

def registrar(etapa, t0, df=None, filas=None):
    """Registra una etapa en el log de monitoreo. t0 = time.time() al inicio de la etapa."""
    duracion = round(time.time() - t0, 2)
    pct_nulos = None
    if df is not None:
        filas = df.count()
        cols = df.columns
        nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).collect()[0]
        pct_nulos = round(sum(v or 0 for v in nulos) / (filas * len(cols)) * 100, 4) if filas else 0.0
    fila = {"run_id": RUN_ID, "origen": ORIGEN, "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "notebook": NOTEBOOK, "etapa": etapa, "duracion_s": duracion, "filas": filas,
            "pct_nulos": pct_nulos, "memoria_mb": memoria_mb(), "cpu_pct": psutil.cpu_percent(interval=0.2),
            "formato": FORMATO}
    pd.DataFrame([fila]).to_csv(LOG_MON, mode="a", index=False, header=not os.path.exists(LOG_MON))
    print(f"⏱️ {etapa}: {duracion} s · filas={filas} · nulos={pct_nulos}% · mem={fila['memoria_mb']} MB")

def mostrar(df, n=20):
    """display() de Databricks no existe en Jupyter/Colab: se muestra como tabla pandas."""
    from IPython.display import display
    display(df.limit(n).toPandas() if hasattr(df, "toPandas") else df)

print(f"Notebook: {NOTEBOOK} · Entorno: {ORIGEN} · Formato de tablas: {FORMATO.upper()} · Spark {spark.version}")
print("LAKE =", LAKE, "| EVID =", EVID)

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML

t0 = time.time()
kpis = leer("oro", "kpis").toPandas().iloc[0]
hora = leer("oro", "fraude_por_hora").toPandas().sort_values("hora")
monto = leer("oro", "fraude_por_monto").toPandas().sort_values("rango_monto")
expo = leer("oro", "exposicion_entidad").toPandas().sort_values("ranking")
tend = leer("oro", "tendencia_sfc").toPandas().sort_values("fecha")
imp = leer("oro", "importancia_variables").toPandas().sort_values("importancia", ascending=False)
umb = leer("oro", "analisis_umbral").toPandas().sort_values("umbral")
met = leer("oro", "metricas_modelo").toPandas().set_index("modelo")
rf = met.loc["Random Forest"]
UMBRAL = json.load(open(f"{EVID}/tablas/modelo_config.json"))["umbral"]
u_sel = umb[umb.umbral.round(2) == round(UMBRAL, 2)].iloc[0]
COLOR_FRAUDE, COLOR_NORMAL, COLOR_GRIS = "#C0392B", "#2E6DA4", "#9AA5B1"
paneles = []      # (titulo, unidad, figura, [lectura1, lectura2])

### V1 · Indicadores clave (KPI)

In [ ]:
fig1 = make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}] * 4])
for i, (val, tit, suf, fmt) in enumerate([
        (kpis.kpi1_tasa_fraude_pct, "KPI1 · Tasa de fraude", " %", ".3f"),
        (kpis.kpi2_monto_fraudado_eur, "KPI2 · Monto fraudado", " €", ",.0f"),
        (u_sel.recall * 100, f"KPI3 · Fraudes detectados (umbral {UMBRAL})", " %", ".1f"),
        (u_sel.precision * 100, "Precisión de las alertas", " %", ".1f")], start=1):
    fig1.add_trace(go.Indicator(mode="number", value=float(val), title={"text": tit},
                                number={"suffix": suf, "valueformat": fmt}), row=1, col=i)
fig1.update_layout(height=230, margin=dict(t=60, b=10))
paneles.append(("V1 · Indicadores clave del caso", "Unidades: %, EUR",
    fig1, [f"De {kpis.transacciones:,.0f} transacciones, {kpis.fraudes:,.0f} fueron fraude ({kpis.kpi1_tasa_fraude_pct:.3f} %), por {kpis.kpi2_monto_fraudado_eur:,.0f} EUR en dos días.",
           f"Con el umbral {UMBRAL}, el modelo detecta {u_sel.recall*100:.1f} % de los fraudes y {u_sel.precision*100:.1f} % de sus alertas son fraude real."]))
fig1.show()

### V2 · Tasa de fraude por hora del día

In [ ]:
fig2 = make_subplots(specs=[[{"secondary_y": True}]])
fig2.add_bar(x=hora.hora, y=hora.transacciones, name="Transacciones", marker_color=COLOR_GRIS)
fig2.add_scatter(x=hora.hora, y=hora.tasa_fraude_pct, name="Tasa de fraude (%)", mode="lines+markers",
                 line=dict(color=COLOR_FRAUDE, width=3), secondary_y=True)
fig2.update_xaxes(title="Hora del día (0–23)", dtick=1)
fig2.update_yaxes(title="Transacciones (n.º)", secondary_y=False)
fig2.update_yaxes(title="Tasa de fraude (%)", secondary_y=True)
fig2.update_layout(height=420, legend=dict(orientation="h", y=1.1))
pico = hora.sort_values("tasa_fraude_pct", ascending=False).iloc[0]
madr = hora[hora.hora <= 5]
paneles.append(("V2 · ¿A qué hora ocurre el fraude?", "Eje izq.: n.º de transacciones · eje der.: % de fraude",
    fig2, [f"La tasa de fraude alcanza su máximo a las {int(pico.hora)}:00 ({pico.tasa_fraude_pct:.3f} %), cuando el volumen de transacciones es mínimo.",
           f"La madrugada (0–5 h) concentra {madr.fraudes.sum()/hora.fraudes.sum()*100:.1f} % de los fraudes con solo {madr.transacciones.sum()/hora.transacciones.sum()*100:.1f} % del tráfico: es la franja a vigilar."]))
fig2.show()

### V3 · Fraude por rango de monto

In [ ]:
fig3 = go.Figure(go.Bar(x=monto.rango_monto.str[3:], y=monto.tasa_fraude_pct, marker_color=COLOR_FRAUDE,
                        text=[f"{int(f)} fraudes" for f in monto.fraudes], textposition="outside"))
fig3.update_layout(height=400, xaxis_title="Rango de monto (EUR)", yaxis_title="Tasa de fraude (%)")
mx = monto.sort_values("tasa_fraude_pct", ascending=False).iloc[0]
mf = monto.sort_values("fraudes", ascending=False).iloc[0]
paneles.append(("V3 · ¿En qué montos se concentra el fraude?", "Tasa de fraude (%) por rango de monto en EUR",
    fig3, [f"El rango {mx.rango_monto[3:]} EUR tiene la mayor tasa ({mx.tasa_fraude_pct:.3f} %); el rango {mf.rango_monto[3:]} EUR concentra más casos ({int(mf.fraudes)}).",
           "Hay fraude en todos los rangos: una regla solo por monto bloquearía clientes legítimos sin detectar la mayoría de fraudes."]))
fig3.show()

### V4 · Exposición estimada por entidad en Colombia (join SFC × F1)

In [ ]:
top = expo.head(10).iloc[::-1]
fig4 = go.Figure(go.Bar(y=top.entidad.str.title(), x=top.tx_compras / 1e6, orientation="h", marker_color=COLOR_NORMAL,
                        text=[f"≈{f:,.0f} fraudes est." for f in top.fraudes_estimados], textposition="outside"))
fig4.update_layout(height=480, xaxis_title="Transacciones de compra con tarjeta de crédito (millones)", margin=dict(l=260))
anio = int(expo.anio.iloc[0])
paneles.append((f"V4 · ¿Dónde está la exposición en Colombia? ({anio})", "Millones de transacciones de compra; etiqueta = fraudes estimados con la tasa de F1",
    fig4, [f"Las 5 entidades con más compras con tarjeta de crédito concentran {expo.head(5).participacion_pct.sum():.1f} % del total nacional de {anio}.",
           f"Aplicando la tasa observada, el escenario estima {expo.fraudes_estimados.sum():,.0f} fraudes en el año: un modelo como este debe desplegarse primero en esas entidades."]))
fig4.show()

### V5 · Tendencia mensual de compras con tarjeta de crédito en Colombia (SFC)

In [ ]:
fig5 = go.Figure()
fig5.add_scatter(x=tend.fecha, y=tend.tx_compras_nal / 1e6, name="Mensual", line=dict(color=COLOR_GRIS))
fig5.add_scatter(x=tend.fecha, y=tend.media_movil_12m / 1e6, name="Media móvil 12 meses", line=dict(color=COLOR_NORMAL, width=3))
fig5.update_layout(height=400, xaxis_title="Mes de corte", yaxis_title="Compras nacionales (millones)",
                   legend=dict(orientation="h", y=1.1))
ult = tend.dropna(subset=["crecimiento_anual_pct"]).iloc[-1]
paneles.append(("V5 · ¿Cuánto crece el volumen transaccional?", "Millones de transacciones de compra nacional por mes",
    fig5, [f"Las compras nacionales pasaron de {tend.tx_compras_nal.iloc[0]/1e6:,.1f} a {tend.tx_compras_nal.iloc[-1]/1e6:,.1f} millones mensuales entre {pd.to_datetime(tend.fecha.iloc[0]).year} y {pd.to_datetime(tend.fecha.iloc[-1]).year}.",
           f"El último mes con comparación anual creció {ult.crecimiento_anual_pct:+.1f} %: el volumen sigue aumentando y con él la superficie de fraude."]))
fig5.show()

### V6 · ¿Por qué? Variables que más explican el fraude

In [ ]:
t10 = imp.head(10).iloc[::-1]
fig6 = go.Figure(go.Bar(y=t10.variable, x=t10.importancia, orientation="h", marker_color=COLOR_FRAUDE))
fig6.update_layout(height=420, xaxis_title="Importancia en el Random Forest (0–1)")
paneles.append(("V6 · ¿Por qué una transacción es fraude?", "Importancia relativa de cada variable (suma = 1)",
    fig6, [f"{', '.join(imp.variable.head(4))} concentran {imp.importancia.head(4).sum()*100:.0f} % de la importancia del modelo.",
           f"El monto (log_amount, {imp.set_index('variable').importancia.get('log_amount', 0)*100:.1f} %) y la hora pesan poco: el fraude se reconoce por el patrón de la transacción, no por cuánto ni cuándo."]))
fig6.show()

### V7 · Precisión vs. recall según el umbral

In [ ]:
fig7 = go.Figure()
for col, color in [("precision", COLOR_NORMAL), ("recall", COLOR_FRAUDE), ("F1", "#27AE60")]:
    fig7.add_scatter(x=umb.umbral, y=umb[col], name=col.capitalize(), mode="lines+markers", line=dict(color=color, width=3))
fig7.add_vline(x=UMBRAL, line_dash="dash", annotation_text=f"umbral elegido {UMBRAL}")
fig7.update_layout(height=400, xaxis_title="Umbral de probabilidad", yaxis_title="Valor de la métrica (0–1)",
                   legend=dict(orientation="h", y=1.1))
u4 = umb[umb.umbral.round(2) == 0.4].iloc[0]
paneles.append(("V7 · ¿Cuántas alertas aceptar?", "Métricas entre 0 y 1 según el umbral",
    fig7, [f"El F1 máximo ({u_sel.F1:.3f}) se logra con umbral {UMBRAL}: {int(u_sel.VP)} fraudes detectados y {int(u_sel.FP)} falsas alarmas.",
           f"Si el banco prioriza no dejar pasar fraudes, el umbral 0,4 detecta {int(u4.VP)} con {int(u4.FP)} falsas alarmas; por debajo de 0,3 las falsas alarmas se disparan."]))
fig7.show()

### V8 · Matriz de confusión del modelo vs. línea base

In [ ]:
fig8 = make_subplots(rows=1, cols=2, subplot_titles=("Línea base (siempre normal)", f"Random Forest (umbral 0,5)"))
for i, nombre in enumerate(["Línea base (siempre normal)", "Random Forest"], start=1):
    r = met.loc[nombre]
    z = [[int(r.VN), int(r.FP)], [int(r.FN), int(r.VP)]]
    fig8.add_trace(go.Heatmap(z=z, x=["Pred. normal", "Pred. fraude"], y=["Real normal", "Real fraude"],
                              text=[[f"{v:,}" for v in fila] for fila in z], texttemplate="%{text}",
                              colorscale="Blues", showscale=False, zmax=200), row=1, col=i)
fig8.update_layout(height=380)
base = met.loc["Línea base (siempre normal)"]
paneles.append(("V8 · ¿Supera el modelo a una regla trivial?", "N.º de transacciones del conjunto de prueba",
    fig8, [f"La línea base acierta {base.exactitud*100:.2f} % pero deja pasar los {int(base.FN)} fraudes; el Random Forest detecta {int(rf.VP)} con solo {int(rf.FP)} falsas alarmas.",
           f"AUC-PR: {rf.AUC_PR:.3f} del modelo frente a {base.AUC_PR:.4f} de la línea base; por eso la exactitud no se usa como métrica en este caso."]))
fig8.show()
registrar("tablero_figuras", t0)

## Publicación del tablero HTML

In [ ]:
t0 = time.time()
partes = [f"""<html><head><meta charset='utf-8'><title>FraudeCero · Tablero</title>
<style>body{{font-family:Segoe UI,Arial,sans-serif;max-width:1200px;margin:auto;padding:24px;color:#1f2d3d;background:#f7f9fb}}
h1{{margin-bottom:4px}} .sub{{color:#5f6b7a;margin-top:0}} .card{{background:white;border-radius:10px;padding:16px 20px;margin:18px 0;box-shadow:0 1px 4px #0001}}
.unidad{{color:#5f6b7a;font-size:13px}} .lectura{{border-left:4px solid #C0392B;padding-left:10px;margin:6px 0}}</style></head><body>
<h1>FraudeCero · Detección de fraude con tarjeta de crédito</h1>
<p class='sub'>Grupo 6 · Big Data Uniciencia 2026-II · Fuentes: Kaggle ULB (F1) y Superintendencia Financiera – datos.gov.co (F2) · Generado: {datetime.datetime.now():%Y-%m-%d %H:%M}</p>"""]
for i, (titulo, unidad, fig, lect) in enumerate(paneles):
    partes.append(f"<div class='card'><h2>{titulo}</h2><div class='unidad'>{unidad}</div>"
                  + fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False)
                  + "".join(f"<p class='lectura'>{l}</p>" for l in lect) + "</div>")
partes.append("</body></html>")
ruta_tablero = f"{EVID}/tablero_fraudecero.html"
with open(ruta_tablero, "w", encoding="utf-8") as f:
    f.write("\n".join(partes))
print("Tablero publicado en:", ruta_tablero, f"({os.path.getsize(ruta_tablero)/1024:.0f} KB) ·", len(paneles), "visualizaciones")
registrar("tablero_html", t0)

## Integración (A3.3): exportación a 2 destinos
| Destino | Formato | Consumidor |
|---|---|---|
| D1 · Archivos | CSV (UTF-8, separador coma) y Parquet en `exportaciones/` | Excel, Power BI Desktop, otros equipos |
| D2 · Base de datos | SQLite `fraudecero.db` (tablas `oro_*`) | Metabase / Power BI (ODBC) / SQL |

In [ ]:
import sqlite3
t0 = time.time()
dir_exp = f"{EVID}/exportaciones"
os.makedirs(dir_exp, exist_ok=True)
tablas_oro = {"kpis": kpis.to_frame().T, "fraude_por_hora": hora, "fraude_por_monto": monto,
              "exposicion_entidad": expo, "tendencia_sfc": tend, "importancia_variables": imp,
              "analisis_umbral": umb, "metricas_modelo": met.reset_index()}
con = sqlite3.connect(f"{dir_exp}/fraudecero.db")
for nombre, df in tablas_oro.items():
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].astype(str)
    df.to_csv(f"{dir_exp}/oro_{nombre}.csv", index=False)                       # D1 · CSV
    df.to_sql(f"oro_{nombre}", con, if_exists="replace", index=False)           # D2 · SQLite
leer("oro", "predicciones").write.mode("overwrite").parquet(f"{LAKE}/exportaciones/oro_predicciones_parquet")  # D1 · Parquet
con.close()
con = sqlite3.connect(f"{dir_exp}/fraudecero.db")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))
print(pd.read_sql("SELECT * FROM oro_kpis", con).T)
con.close()
registrar("integracion_exportacion", t0)

## Contrato de datos (A3.3)

In [ ]:
contrato = pd.DataFrame([
    ("oro_kpis", "transacciones", "entero", "Sí", "Por ejecución del pipeline", "Juan David Montoya"),
    ("oro_kpis", "fraudes", "entero", "Sí", "Por ejecución del pipeline", "Juan David Montoya"),
    ("oro_kpis", "kpi1_tasa_fraude_pct", "decimal(4) %", "Sí", "Por ejecución del pipeline", "Juan David Montoya"),
    ("oro_kpis", "kpi2_monto_fraudado_eur", "decimal(2) EUR", "Sí", "Por ejecución del pipeline", "Juan David Montoya"),
    ("oro_fraude_por_hora", "hora", "entero 0–23", "Sí", "Por ejecución del pipeline", "William Conde Villamizar"),
    ("oro_fraude_por_hora", "tasa_fraude_pct", "decimal(4) %", "Sí", "Por ejecución del pipeline", "William Conde Villamizar"),
    ("oro_fraude_por_hora", "ranking_riesgo", "entero ≥ 1", "Sí", "Por ejecución del pipeline", "William Conde Villamizar"),
    ("oro_exposicion_entidad", "entidad", "texto (mayúsculas)", "Sí", "Mensual (corte SFC)", "William Conde Villamizar"),
    ("oro_exposicion_entidad", "tx_compras", "entero", "Sí", "Mensual (corte SFC)", "William Conde Villamizar"),
    ("oro_exposicion_entidad", "fraudes_estimados", "entero (escenario)", "Sí", "Mensual (corte SFC)", "William Conde Villamizar"),
    ("oro_predicciones", "id_hash", "texto SHA-256 (64)", "Sí", "Por reentrenamiento", "Juan David Montoya"),
    ("oro_predicciones", "p_fraude", "decimal 0–1", "Sí", "Por reentrenamiento", "Juan David Montoya"),
    ("oro_predicciones", "prediccion_umbral", "entero 0/1", "Sí", "Por reentrenamiento", "Juan David Montoya"),
    ("oro_metricas_modelo", "AUC_PR, recall, precision, F1", "decimal 0–1", "Sí", "Por reentrenamiento", "Juan David Montoya"),
], columns=["tabla", "campo", "tipo", "obligatorio", "frecuencia", "responsable"])
contrato.to_csv(f"{EVID}/tablas/contrato_datos.csv", index=False)
mostrar(contrato, 30)

## Cierre
Respalda el Lakehouse (en Colab) y confirma la ejecución completa.

In [ ]:
respaldar_lake()
print("✅ Notebook", NOTEBOOK, "terminado · run_id:", RUN_ID)

## RESUMEN

In [ ]:
print("=== C1 · KPI ===")
print(kpis.to_string())

print("\n=== C2 · Por hora ===")
h = hora.sort_values("ranking_riesgo").head(5)
print(h[["hora", "transacciones", "fraudes", "tasa_fraude_pct"]].to_string(index=False))
m = hora[hora.hora <= 5]
print(f"Madrugada 0-5h: {m.fraudes.sum()/hora.fraudes.sum()*100:.1f} % de fraudes · {m.transacciones.sum()/hora.transacciones.sum()*100:.1f} % del tráfico")

print("\n=== C3 · Por monto ===")
print(monto[["rango_monto", "transacciones", "fraudes", "tasa_fraude_pct", "pct_de_los_fraudes"]].to_string(index=False))

print("\n=== C4 · Entidades ===")
print(expo.head(5)[["anio", "entidad", "tx_compras", "participacion_pct", "fraudes_estimados"]].to_string(index=False))
print(f"Top 5 = {expo.head(5).participacion_pct.sum():.1f} % · entidades para el 80 %: {int((expo.participacion_acum_pct < 80).sum()) + 1} · fraudes estimados en el año: {expo.fraudes_estimados.sum():,.0f}")

print("\n=== C5 · Tendencia SFC ===")
print(tend.dropna(subset=["crecimiento_anual_pct"]).tail(1)[["fecha", "tx_compras_nal", "crecimiento_anual_pct"]].to_string(index=False))

print("\n=== Métricas del modelo ===")
print(met[["AUC_PR", "precision", "recall", "F1", "VP", "FP", "FN", "VN"]].to_string())
print(umb.to_string(index=False))

print("\n=== Optimización ===")
print(pd.read_csv(f"{EVID}/tablas/optimizacion_tiempos.csv").to_string(index=False))

print("\n=== Monitoreo ===")
log = pd.read_csv(LOG_MON)
print("Ejecuciones registradas:", log.run_id.nunique(), "·", sorted(log.run_id.unique()))